# Transfer Learning with Deep Tabular Models (ICLR 2023) — tái hiện pipeline trên Google Colab

Notebook này gói toàn bộ pipeline của bài báo (Levin et al., 2023; repo gốc: `github.com/LevinRoman/tabular-transfer-learning`):

1. **Pre-train** mô hình sâu (FT-Transformer, ResNet, MLP, TabTransformer) đa nhãn trên K−1 nhãn upstream.
2. **Fine-tune** trên nhãn downstream còn lại với 4, 10, 20, 100, 200 mẫu theo 4 kịch bản: `LH`, `MLP` (feature extractor đóng băng), `LH-E2E`, `MLP-E2E` (fine-tune toàn bộ).
3. So sánh với **From Scratch (FS)**, **CatBoost/XGBoost** và **GBDT + Stacking**.
4. Xếp hạng bằng kiểm định **Wilcoxon rank-sum một phía (p = 0.05)** và vẽ heatmap giống Hình 2.

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU` (FT-Transformer rất chậm trên CPU).

Chạy lần lượt các ô từ trên xuống. Phần 3 có 3 lựa chọn dữ liệu, chỉ cần chạy **một** trong số đó.

## 1. Kiểm tra GPU và cài thư viện

In [ ]:
!nvidia-smi -L || echo "Không có GPU - vẫn chạy được nhưng chậm"
!pip install -q optuna catboost xgboost
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

### (Khuyến nghị) Gắn Google Drive để lưu kết quả
Phiên Colab có thể bị ngắt. Lưu kết quả lên Drive thì khi chạy lại **cùng lệnh**, code tự bỏ qua phần đã xong (cache siêu tham số, trọng số pre-train, kết quả từng seed).

In [ ]:
USE_DRIVE = True   # đổi thành False nếu không muốn dùng Drive

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/tabular_tl_repro'
else:
    WORK = '/content/tabular_tl_repro'
os.makedirs(WORK, exist_ok=True)
%cd $WORK
print("Thư mục làm việc:", WORK)

## 2. Ghi mã nguồn ra file
Mỗi ô dưới đây tạo một file `.py` trong thư mục làm việc. Chạy hết các ô này (không cần sửa).

| File | Vai trò |
|---|---|
| `config.py` | Hằng số, siêu tham số mặc định (Bảng 2–5), lịch huấn luyện |
| `data.py` | Đọc dữ liệu, chia 65/15/20, Quantile + điền mean, rút mẫu downstream |
| `models.py` | FT-Transformer, ResNet, MLP, TabTransformer; thay head, đóng băng |
| `training.py` | Pre-train, fine-tune 4 kịch bản, From Scratch |
| `gbdt.py` | CatBoost / XGBoost và Stacking |
| `tuning.py` | Không gian tìm kiếm Optuna (Bảng 2–7) |
| `run_experiments.py` | Vòng lặp thực nghiệm chính |
| `analyze_results.py` | Xếp hạng có ý nghĩa thống kê + heatmap |
| `make_synthetic_data.py` | Sinh dữ liệu giả cấu trúc MetaMIMIC |

In [ ]:
%%writefile config.py
"""
config.py
Cấu hình thực nghiệm, bám theo bài báo "Transfer Learning with Deep Tabular Models"
(Levin et al., ICLR 2023) và repo chính thức github.com/LevinRoman/tabular-transfer-learning.
"""
from dataclasses import dataclass, field
from typing import List

# 12 nhiệm vụ chẩn đoán của MetaMIMIC (đúng tên cột trong repo gốc, mimic_tools.py)
MIMIC_TARGETS = [
    "diabetes_diagnosed", "hypertensive_diagnosed", "ischematic_diagnosed",
    "heart_diagnosed", "overweight_diagnosed", "anemia_diagnosed",
    "respiratory_diagnosed", "hypotension_diagnosed", "lipoid_diagnosed",
    "atrial_diagnosed", "purpura_diagnosed", "alcohol_diagnosed",
]

DEEP_MODELS = ["ft_transformer", "resnet", "mlp", "tab_transformer"]
GBDT_MODELS = ["catboost", "xgboost"]

# 4 kịch bản transfer (Mục 3.3) + baseline From Scratch
TRANSFER_SETUPS = ["LH", "MLP", "LH-E2E", "MLP-E2E"]   # LH = linear head, E2E = fine-tune toàn bộ

# Giá trị mặc định của các mô hình (cột "Default" trong Bảng 2-5, Phụ lục C)
DEFAULT_HPARAMS = {
    "ft_transformer": dict(n_layers=3, d_token=192, residual_dropout=0.0, attention_dropout=0.2,
                           ffn_dropout=0.1, d_ffn_factor=4 / 3, n_heads=8,
                           lr=1e-4, weight_decay=1e-5),
    "resnet": dict(n_layers=5, d_embedding=128, d=200, d_hidden_factor=3.0, hidden_dropout=0.2,
                   residual_dropout=0.2, lr=1e-4, weight_decay=0.0),
    "mlp": dict(d_embedding=128, d_layers=[300, 200, 300], dropout=0.2, lr=1e-4, weight_decay=1e-5),
    "tab_transformer": dict(n_heads=8, n_layers=6, d_embedding=32, attention_dropout=0.0,
                            ff_dropout=0.0, lr=1e-4, weight_decay=1e-5),
    "catboost": dict(),   # tham số mặc định của thư viện
    "xgboost": dict(),
}


@dataclass
class ExpConfig:
    # --- Chia dữ liệu (Phụ lục B.4; repo: random_state=1, 0.2 test rồi 0.1875 của phần còn lại = 15%)
    test_size: float = 0.20
    val_size_of_rest: float = 0.1875
    split_seed: int = 1
    downstream_sample_seed: int = 0          # repo cố định seed=0 khi rút mẫu downstream
    sample_sizes: List[int] = field(default_factory=lambda: [4, 10, 20, 100, 200])

    # --- Seeds & tuning (Mục 3.4, Phụ lục B.4, C)
    seeds: List[int] = field(default_factory=lambda: list(range(10)))
    n_trials: int = 50
    tune: bool = True
    pretrain_per_seed: bool = False   # False: pre-train 1 lần (seed 0) rồi fine-tune với 10 seeds

    # --- Pre-training (Phụ lục B.6.1)
    pretrain_epochs: int = 500
    pretrain_patience: int = 30
    batch_size: int = 256

    # --- Fine-tuning / from scratch (Phụ lục B.6.1, B.6.3)
    finetune_lr: float = 5e-5          # lr cho feature extractor khi fine-tune
    head_lr: float = 1e-3              # lr của head (repo: hyp_for_transfer.yaml -> head_lr)
    head_warmup_epochs: int = 10       # repo: head_warmup_period (đóng băng FE 10 epoch đầu khi E2E)
    scratch_lr: float = 1e-4
    max_downstream_epochs: int = 200
    downstream_patience: int = 30
    downstream_val_frac: float = 0.20  # với 100/200 mẫu: tách 20% làm val để early stopping (E2E)
    mlp_head_dim: int = 200            # head MLP 2 lớp x 200 neuron (Mục 3.3)

    # --- Đánh giá (Phụ lục B.6.5)
    alpha: float = 0.05                # Wilcoxon rank-sum một phía


def quick_config() -> ExpConfig:
    """Cấu hình rút gọn để chạy thử nhanh (smoke test) – KHÔNG dùng để báo cáo kết quả."""
    return ExpConfig(sample_sizes=[4, 20, 100], seeds=[0, 1, 2], n_trials=2,
                     pretrain_epochs=15, pretrain_patience=5, max_downstream_epochs=40)

In [ ]:
%%writefile data.py
"""
data.py
- Đọc MetaMIMIC (hoặc Yeast/Emotions dạng ARFF/CSV của Mulan)
- Chia 65% upstream-train / 15% upstream-val / 20% downstream-test (cố định qua các seed)
- Tiền xử lý: điền mean, QuantileTransformer (normal) fit trên upstream rồi áp cho downstream,
  biến phân loại thiếu -> hạng mục mới
- Rút mẫu downstream nhỏ (4/10/20/100/200) có phân tầng, đảm bảo đủ 2 lớp
"""
from dataclasses import dataclass
from typing import List, Optional

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, QuantileTransformer

from config import MIMIC_TARGETS, ExpConfig  # noqa


# ----------------------------------------------------------------------------- loaders
def load_metamimic(path: str):
    """MetaMIMIC.csv: subject_id, 171 cột số, gender, 12 cột *_diagnosed."""
    df = pd.read_csv(path)
    targets = [c for c in MIMIC_TARGETS if c in df.columns]
    if len(targets) != 12:
        raise ValueError(f"Chỉ tìm thấy {len(targets)}/12 cột mục tiêu MetaMIMIC trong {path}")
    Y = df[targets].astype(int)
    X = df.drop(columns=targets + [c for c in ["subject_id"] if c in df.columns])
    cat_cols = [c for c in ["gender"] if c in X.columns]
    num_cols = [c for c in X.columns if c not in cat_cols]
    return X, Y, num_cols, cat_cols


def load_mulan(path: str, n_labels: int):
    """Yeast (14 nhãn) / Emotions (6 nhãn) theo định dạng Mulan: n_labels cột cuối là nhãn.
    Hỗ trợ .arff (scipy) hoặc .csv."""
    if path.endswith(".arff"):
        from scipy.io import arff
        data, _ = arff.loadarff(path)
        df = pd.DataFrame(data)
    else:
        df = pd.read_csv(path)
    label_cols = list(df.columns[-n_labels:])
    Y = df[label_cols].apply(lambda s: s.map(lambda v: int(v.decode() if isinstance(v, bytes) else v)))
    X = df.drop(columns=label_cols).astype(float)
    return X, Y, list(X.columns), []


def load_openml_yeast():
    """Yeast (2417 mẫu, 103 đặc trưng, 14 nhãn) tải trực tiếp từ OpenML (https://www.openml.org/d/40597)."""
    from sklearn.datasets import fetch_openml
    X, Y = fetch_openml("yeast", version=4, return_X_y=True, as_frame=True, parser="auto")
    Y = (Y.astype(str).apply(lambda s: s.str.upper()) == "TRUE").astype(int)
    X = X.astype(float)
    return X, Y, list(X.columns), []


# ----------------------------------------------------------------------------- containers
@dataclass
class TabularData:
    x_num: np.ndarray            # đặc trưng số đã chuẩn hoá (cho mạng neuron)
    x_cat: Optional[np.ndarray]  # chỉ số hạng mục (int64) hoặc None
    x_gbdt: np.ndarray           # đặc trưng gốc (đã điền mean) cho GBDT
    y: np.ndarray                # (N,) cho nhị phân, (N, K) cho đa nhãn

    def __len__(self):
        return len(self.y)

    def subset(self, idx):
        return TabularData(self.x_num[idx], None if self.x_cat is None else self.x_cat[idx],
                           self.x_gbdt[idx], self.y[idx])


def stratified_sample(y: np.ndarray, n: int, seed: int) -> np.ndarray:
    """Rút n chỉ số có phân tầng theo y, đảm bảo mỗi lớp có ít nhất 1 mẫu
    (giống stratified_sample_at_least_one_per_class trong repo)."""
    idx_all = np.arange(len(y))
    if n >= len(y):
        return idx_all
    rng = np.random.RandomState(seed)
    classes = np.unique(y)
    try:
        idx, _ = train_test_split(idx_all, train_size=n, stratify=y, random_state=seed)
    except ValueError:
        idx = np.array([], dtype=int)
    if len(np.unique(y[idx])) == len(classes) and len(idx) == n:
        return np.sort(idx)
    # fallback: 1 mẫu/lớp + phần còn lại
    one_each = np.array([rng.choice(idx_all[y == c]) for c in classes])
    rest_pool = np.setdiff1d(idx_all, one_each)
    k = n - len(one_each)
    if k <= 0:
        return np.sort(one_each[:n])
    try:
        rest, _ = train_test_split(rest_pool, train_size=k, stratify=y[rest_pool], random_state=seed)
    except ValueError:
        rest = rng.choice(rest_pool, size=k, replace=False)
    return np.sort(np.concatenate([one_each, rest]))


# ----------------------------------------------------------------------------- benchmark
class TransferBenchmark:
    """Giữ toàn bộ dữ liệu đã tiền xử lý và tạo các bài toán upstream/downstream."""

    def __init__(self, X: pd.DataFrame, Y: pd.DataFrame, num_cols: List[str], cat_cols: List[str],
                 cfg: ExpConfig, name: str = "metamimic"):
        self.name, self.cfg = name, cfg
        self.target_names = list(Y.columns)
        self.n_targets = len(self.target_names)
        X = X.reset_index(drop=True)
        self.Y = Y.reset_index(drop=True).values.astype(np.float32)

        idx = np.arange(len(X))
        rest, self.idx_test = train_test_split(idx, test_size=cfg.test_size, random_state=cfg.split_seed)
        self.idx_train, self.idx_val = train_test_split(rest, test_size=cfg.val_size_of_rest,
                                                        random_state=cfg.split_seed)
        # Pool cho downstream: upstream train + val (repo gộp val vào train ở stage downstream)
        self.idx_pool = np.concatenate([self.idx_train, self.idx_val])

        # --- số: điền mean (tính trên upstream train)
        Xn = X[num_cols].astype(float)
        means = Xn.iloc[self.idx_train].mean()
        Xn = Xn.fillna(means).fillna(0.0).values
        # --- QuantileTransformer fit trên upstream train (Phụ lục B.5), noise nhỏ như repo
        tr = Xn[self.idx_train].copy()
        stds = np.std(tr, axis=0, keepdims=True)
        tr += (1e-3 / np.maximum(stds, 1e-3)) * np.random.default_rng(0).standard_normal(tr.shape)
        qt = QuantileTransformer(output_distribution="normal",
                                 n_quantiles=max(min(len(tr) // 30, 1000), 10),
                                 subsample=int(1e9), random_state=0)
        qt.fit(tr)
        self.x_num = qt.transform(Xn).astype(np.float32)

        # --- phân loại: NaN -> hạng mục mới, mã hoá ordinal
        if cat_cols:
            Xc = X[cat_cols].astype(object).where(X[cat_cols].notna(), "MissingValue").astype(str)
            enc = OrdinalEncoder(dtype=np.int64).fit(Xc.values)
            self.x_cat = enc.transform(Xc.values)
            self.cat_cardinalities = [len(c) for c in enc.categories_]
        else:
            self.x_cat, self.cat_cardinalities = None, None

        # --- GBDT dùng đặc trưng gốc (không chuẩn hoá)
        self.x_gbdt = Xn.astype(np.float32) if self.x_cat is None else \
            np.hstack([Xn, self.x_cat]).astype(np.float32)
        self.n_num = self.x_num.shape[1]

        print(f"[{name}] N={len(X)} | upstream-train={len(self.idx_train)} "
              f"| upstream-val={len(self.idx_val)} | test={len(self.idx_test)} "
              f"| n_num={self.n_num} | cat={self.cat_cardinalities}")

    # ------------------------------------------------------------------ helpers
    def _make(self, idx, y):
        return TabularData(self.x_num[idx], None if self.x_cat is None else self.x_cat[idx],
                           self.x_gbdt[idx], y)

    def upstream_targets(self, downstream_target: int) -> List[int]:
        return [t for t in range(self.n_targets) if t != downstream_target]

    def upstream(self, downstream_target: int):
        """Bài toán pre-training đa nhãn với K-1 nhãn còn lại."""
        cols = self.upstream_targets(downstream_target)
        return {"train": self._make(self.idx_train, self.Y[self.idx_train][:, cols]),
                "val": self._make(self.idx_val, self.Y[self.idx_val][:, cols])}

    def upstream_single(self, target: int, n: int, seed: int):
        """Dữ liệu proxy để tune siêu tham số cho FS/GBDT: 1 nhãn upstream, n mẫu train,
        val = toàn bộ upstream-val (Mục 3.4)."""
        y_tr = self.Y[self.idx_train, target]
        sub = self.idx_train[stratified_sample(y_tr, n, seed)]
        return {"train": self._make(sub, self.Y[sub, target]),
                "val": self._make(self.idx_val, self.Y[self.idx_val, target])}

    def downstream(self, target: int, n: int):
        """Bài toán downstream nhị phân với n mẫu huấn luyện; test = 20% cố định."""
        y_pool = self.Y[self.idx_pool, target]
        sub = self.idx_pool[stratified_sample(y_pool, n, self.cfg.downstream_sample_seed)]
        return {"train": self._make(sub, self.Y[sub, target]),
                "test": self._make(self.idx_test, self.Y[self.idx_test, target]),
                "train_idx": sub}

In [ ]:
%%writefile models.py
"""
models.py
FT-Transformer, ResNet, MLP (chuyển thể từ RTDL / repo gốc) và TabTransformer.
Mọi mô hình có: features(x_num, x_cat) -> biểu diễn; self.head -> logits.
Nhờ vậy có thể thay head (linear / MLP 2x200) và đóng băng feature extractor.
"""
import math
from typing import List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


class CatEmbedding(nn.Module):
    def __init__(self, cardinalities: List[int], d: int):
        super().__init__()
        self.register_buffer("offsets", torch.tensor([0] + cardinalities[:-1]).cumsum(0))
        self.emb = nn.Embedding(sum(cardinalities), d)
        nn.init.kaiming_uniform_(self.emb.weight, a=math.sqrt(5))

    def forward(self, x_cat):              # -> (B, n_cat, d)
        return self.emb(x_cat + self.offsets[None])


class TabularModel(nn.Module):
    head: nn.Module

    def features(self, x_num, x_cat):
        raise NotImplementedError

    def forward(self, x_num, x_cat=None):
        return self.head(self.features(x_num, x_cat))


# ============================================================================ MLP
class MLP(TabularModel):
    def __init__(self, n_num, cards, d_out, d_layers, dropout, d_embedding):
        super().__init__()
        d_in = n_num
        self.cat = None
        if cards:
            self.cat = CatEmbedding(cards, d_embedding)
            d_in += len(cards) * d_embedding
        self.layers = nn.ModuleList([nn.Linear(d_layers[i - 1] if i else d_in, d) for i, d in enumerate(d_layers)])
        self.dropout = dropout
        self.head = nn.Linear(d_layers[-1] if d_layers else d_in, d_out)

    def features(self, x_num, x_cat):
        x = [x_num]
        if self.cat is not None and x_cat is not None:
            x.append(self.cat(x_cat).flatten(1))
        x = torch.cat(x, dim=-1)
        for layer in self.layers:
            x = F.dropout(F.relu(layer(x)), self.dropout, self.training)
        return x


# ============================================================================ ResNet
class ResNet(TabularModel):
    def __init__(self, n_num, cards, d_out, d, d_hidden_factor, n_layers, hidden_dropout,
                 residual_dropout, d_embedding):
        super().__init__()
        d_in = n_num
        self.cat = None
        if cards:
            self.cat = CatEmbedding(cards, d_embedding)
            d_in += len(cards) * d_embedding
        d_hidden = int(d * d_hidden_factor)
        self.first_layer = nn.Linear(d_in, d)
        self.layers = nn.ModuleList([nn.ModuleDict({
            "norm": nn.BatchNorm1d(d), "linear0": nn.Linear(d, d_hidden), "linear1": nn.Linear(d_hidden, d)})
            for _ in range(n_layers)])
        self.last_norm = nn.BatchNorm1d(d)
        self.hidden_dropout, self.residual_dropout = hidden_dropout, residual_dropout
        self.head = nn.Linear(d, d_out)

    def features(self, x_num, x_cat):
        x = [x_num]
        if self.cat is not None and x_cat is not None:
            x.append(self.cat(x_cat).flatten(1))
        x = self.first_layer(torch.cat(x, dim=-1))
        for layer in self.layers:
            z = F.relu(layer["linear0"](layer["norm"](x)))
            z = F.dropout(z, self.hidden_dropout, self.training)
            z = F.dropout(layer["linear1"](z), self.residual_dropout, self.training)
            x = x + z
        return F.relu(self.last_norm(x))


# ============================================================================ FT-Transformer
def reglu(x):
    a, b = x.chunk(2, dim=-1)
    return a * F.relu(b)


class FTTokenizer(nn.Module):
    def __init__(self, n_num, cards, d_token):
        super().__init__()
        self.cat = CatEmbedding(cards, d_token) if cards else None
        n_bias = n_num + (len(cards) if cards else 0)
        self.weight = nn.Parameter(torch.empty(n_num + 1, d_token))   # +1 cho [CLS]
        self.bias = nn.Parameter(torch.empty(n_bias, d_token))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.bias, a=math.sqrt(5))

    def forward(self, x_num, x_cat):
        x_num = torch.cat([torch.ones(len(x_num), 1, device=x_num.device), x_num], dim=1)
        x = self.weight[None] * x_num[:, :, None]
        if self.cat is not None and x_cat is not None:
            x = torch.cat([x, self.cat(x_cat)], dim=1)
        bias = torch.cat([torch.zeros(1, self.bias.shape[1], device=x.device), self.bias])
        return x + bias[None]


class MHA(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.W_q, self.W_k, self.W_v = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.W_out = nn.Linear(d, d) if n_heads > 1 else None
        for m in [self.W_q, self.W_k, self.W_v]:
            nn.init.zeros_(m.bias)
        if self.W_out is not None:
            nn.init.zeros_(self.W_out.bias)
        self.n_heads, self.dropout = n_heads, nn.Dropout(dropout) if dropout else None

    def _split(self, x):
        b, t, d = x.shape
        return x.reshape(b, t, self.n_heads, d // self.n_heads).transpose(1, 2)

    def forward(self, x_q, x_kv):
        q, k, v = self._split(self.W_q(x_q)), self._split(self.W_k(x_kv)), self._split(self.W_v(x_kv))
        att = F.softmax(q @ k.transpose(-1, -2) / math.sqrt(q.shape[-1]), dim=-1)
        if self.dropout is not None:
            att = self.dropout(att)
        x = (att @ v).transpose(1, 2).reshape(x_q.shape[0], x_q.shape[1], -1)
        return self.W_out(x) if self.W_out is not None else x


class FTTransformer(TabularModel):
    def __init__(self, n_num, cards, d_out, d_token, n_layers, n_heads, d_ffn_factor,
                 attention_dropout, ffn_dropout, residual_dropout):
        super().__init__()
        self.tokenizer = FTTokenizer(n_num, cards, d_token)
        d_hidden = int(d_token * d_ffn_factor)
        self.layers = nn.ModuleList()
        for i in range(n_layers):
            layer = nn.ModuleDict({"attention": MHA(d_token, n_heads, attention_dropout),
                                   "linear0": nn.Linear(d_token, d_hidden * 2),   # ReGLU
                                   "linear1": nn.Linear(d_hidden, d_token),
                                   "norm1": nn.LayerNorm(d_token)})
            if i:   # prenormalization: lớp đầu không có norm0
                layer["norm0"] = nn.LayerNorm(d_token)
            self.layers.append(layer)
        self.last_norm = nn.LayerNorm(d_token)
        self.ffn_dropout, self.residual_dropout = ffn_dropout, residual_dropout
        self.head = nn.Linear(d_token, d_out)

    def features(self, x_num, x_cat):
        x = self.tokenizer(x_num, x_cat)
        for i, layer in enumerate(self.layers):
            last = i + 1 == len(self.layers)
            r = layer["norm0"](x) if "norm0" in layer else x
            r = layer["attention"](r[:, :1] if last else r, r)   # lớp cuối chỉ cần [CLS]
            if last:
                x = x[:, :1]
            x = x + F.dropout(r, self.residual_dropout, self.training)
            r = reglu(layer["linear0"](layer["norm1"](x)))
            r = layer["linear1"](F.dropout(r, self.ffn_dropout, self.training))
            x = x + F.dropout(r, self.residual_dropout, self.training)
        return F.relu(self.last_norm(x[:, 0]))


# ============================================================================ TabTransformer
class TTBlock(nn.Module):
    def __init__(self, d, n_heads, attn_dropout, ff_dropout):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=attn_dropout, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(ff_dropout), nn.Linear(4 * d, d))

    def forward(self, x):
        h = self.n1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        return x + self.ff(self.n2(x))


class TabTransformer(TabularModel):
    """Transformer cho đặc trưng phân loại + LayerNorm cho đặc trưng số, nối lại rồi qua MLP."""

    def __init__(self, n_num, cards, d_out, d_embedding, n_layers, n_heads, attention_dropout, ff_dropout):
        super().__init__()
        self.cat = None
        n_cat = len(cards) if cards else 0
        if n_cat:
            self.cat = CatEmbedding(cards, d_embedding)
            self.blocks = nn.ModuleList([TTBlock(d_embedding, n_heads, attention_dropout, ff_dropout)
                                         for _ in range(n_layers)])
        self.num_norm = nn.LayerNorm(n_num)
        d_in = n_num + n_cat * d_embedding
        self.mlp = nn.Sequential(nn.Linear(d_in, 4 * d_in), nn.ReLU(), nn.Linear(4 * d_in, 2 * d_in), nn.ReLU())
        self.head = nn.Linear(2 * d_in, d_out)

    def features(self, x_num, x_cat):
        parts = [self.num_norm(x_num)]
        if self.cat is not None and x_cat is not None:
            h = self.cat(x_cat)
            for b in self.blocks:
                h = b(h)
            parts.append(h.flatten(1))
        return self.mlp(torch.cat(parts, dim=-1))


# ============================================================================ factory
def build_model(name: str, n_num: int, cards: Optional[List[int]], d_out: int, hp: dict) -> TabularModel:
    if name == "mlp":
        return MLP(n_num, cards, d_out, list(hp["d_layers"]), hp["dropout"], hp["d_embedding"])
    if name == "resnet":
        return ResNet(n_num, cards, d_out, hp["d"], hp["d_hidden_factor"], hp["n_layers"],
                      hp["hidden_dropout"], hp["residual_dropout"], hp["d_embedding"])
    if name == "ft_transformer":
        return FTTransformer(n_num, cards, d_out, hp["d_token"], hp["n_layers"], hp.get("n_heads", 8),
                             hp["d_ffn_factor"], hp["attention_dropout"], hp["ffn_dropout"],
                             hp["residual_dropout"])
    if name == "tab_transformer":
        d = int(math.ceil(hp["d_embedding"] / hp["n_heads"]) * hp["n_heads"])   # chia hết cho số head
        return TabTransformer(n_num, cards, d_out, d, hp["n_layers"], hp["n_heads"],
                              hp["attention_dropout"], hp["ff_dropout"])
    raise ValueError(name)


def replace_head(model: TabularModel, d_out: int, mlp_head: bool, hidden: int = 200):
    """Khởi tạo lại head: linear hoặc MLP 2 lớp x 200 neuron (giống load_transfer_model_from_checkpoint)."""
    d_in = model.head.in_features if isinstance(model.head, nn.Linear) else model.head[0].in_features
    model.head = (nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                                nn.Linear(hidden, d_out)) if mlp_head else nn.Linear(d_in, d_out))
    return model


def freeze_feature_extractor(model: TabularModel):
    for n, p in model.named_parameters():
        p.requires_grad = n.startswith("head.")
    model._frozen = True
    return model

In [ ]:
%%writefile training.py
"""
training.py
Vòng lặp huấn luyện chung + 3 quy trình:
  pretrain()      : multi-label trên upstream (AdamW, tối đa 500 epoch, patience 30)
  finetune()      : 4 kịch bản LH / MLP / LH-E2E / MLP-E2E (Mục 3.3, Phụ lục B.6.3)
  train_scratch() : baseline From Scratch trên downstream
"""
import copy
import random

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from config import ExpConfig
from data import TabularData
from models import build_model, freeze_feature_extractor, replace_head


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _tensors(d: TabularData, device):
    xn = torch.as_tensor(d.x_num, device=device)
    xc = None if d.x_cat is None else torch.as_tensor(d.x_cat, device=device)
    return xn, xc


@torch.no_grad()
def predict_logits(model, d: TabularData, device, bs: int = 256) -> np.ndarray:
    model.eval()
    xn, xc = _tensors(d, device)
    out = [model(xn[i:i + bs], None if xc is None else xc[i:i + bs]).float().cpu() for i in range(0, len(d), bs)]
    out = torch.cat(out).numpy()
    return out[:, 0] if (out.ndim == 2 and out.shape[1] == 1 and d.y.ndim == 1) else out


def auc(y: np.ndarray, scores: np.ndarray) -> float:
    """ROC-AUC; với đa nhãn lấy trung bình macro trên các nhãn có đủ 2 lớp."""
    if y.ndim == 1:
        return float(roc_auc_score(y, scores)) if len(np.unique(y)) > 1 else float("nan")
    vals = [roc_auc_score(y[:, j], scores[:, j]) for j in range(y.shape[1]) if len(np.unique(y[:, j])) > 1]
    return float(np.mean(vals)) if vals else float("nan")


def fit(model, train: TabularData, val: TabularData = None, *, epochs: int, lr: float, weight_decay: float,
        device, seed: int, batch_size: int = 256, patience: int = None, head_lr: float = None,
        head_warmup_epochs: int = 0):
    """Huấn luyện bằng AdamW + BCEWithLogits. Nếu có val: chọn epoch tốt nhất theo ROC-AUC
    và dừng sớm theo patience. Trả về (model, số epoch tốt nhất, điểm val tốt nhất)."""
    set_seed(seed)
    model.to(device)
    frozen = getattr(model, "_frozen", False)
    fe = [p for n, p in model.named_parameters() if not n.startswith("head.") and p.requires_grad]
    hd = [p for n, p in model.named_parameters() if n.startswith("head.") and p.requires_grad]
    groups = ([{"params": fe, "lr": lr}] if fe else []) + [{"params": hd, "lr": head_lr or lr}]
    opt = torch.optim.AdamW(groups, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()

    xn, xc = _tensors(train, device)
    y = torch.as_tensor(train.y, device=device)
    n = len(train)
    g = torch.Generator().manual_seed(seed)
    best_score, best_epoch, best_state = -np.inf, 0, None

    for ep in range(epochs):
        if fe and head_warmup_epochs:          # repo: lr của feature extractor = 0 trong giai đoạn warm-up head
            opt.param_groups[0]["lr"] = 0.0 if ep < head_warmup_epochs else lr
        model.train()
        if frozen:                              # FE đóng băng: giữ eval mode (dropout/BN cố định)
            for name, m in model.named_children():
                if name != "head":
                    m.eval()
        perm = torch.randperm(n, generator=g).to(device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            if len(idx) == 1 and n > 1:         # tránh lỗi BatchNorm với batch 1 mẫu
                continue
            out = model(xn[idx], None if xc is None else xc[idx])
            target = y[idx]
            if target.ndim == 1:
                out = out.squeeze(-1)
            loss = loss_fn(out, target)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        if val is not None:
            score = auc(val.y, predict_logits(model, val, device))
            if score > best_score:
                best_score, best_epoch, best_state = score, ep + 1, copy.deepcopy(model.state_dict())
            elif patience is not None and (ep + 1 - best_epoch) > patience:
                break
    if val is not None and best_state is not None:
        model.load_state_dict(best_state)
    else:
        best_epoch = epochs
    return model, best_epoch, best_score


# ============================================================================ pre-training
def pretrain(arch: str, hp: dict, up: dict, cards, cfg: ExpConfig, device, seed: int = 0):
    d_out = up["train"].y.shape[1]
    model = build_model(arch, up["train"].x_num.shape[1], cards, d_out, hp)
    model, best_ep, val_auc = fit(model, up["train"], up["val"], epochs=cfg.pretrain_epochs, lr=hp["lr"],
                                  weight_decay=hp["weight_decay"], patience=cfg.pretrain_patience,
                                  batch_size=cfg.batch_size, device=device, seed=seed)
    return model, val_auc, best_ep


# ============================================================================ fine-tuning
def downstream_schedule(n: int, setup: str):
    """Chọn số epoch theo Phụ lục B.6.3. Trả về (epochs, dùng_val_early_stopping)."""
    e2e, mlp_head = setup.endswith("E2E"), setup.startswith("MLP")
    if n <= 10:
        return 30, False
    if n <= 20:
        return 60, False
    if e2e or setup == "FS":
        return None, True                      # 20% val + early stopping (patience 30)
    return (100 if mlp_head else 200), False


def _split_val(train: TabularData, frac: float, seed: int):
    idx = np.arange(len(train))
    try:
        tr, va = train_test_split(idx, test_size=frac, stratify=train.y, random_state=seed)
    except ValueError:
        tr, va = train_test_split(idx, test_size=frac, random_state=seed)
    return train.subset(tr), train.subset(va)


def finetune(arch: str, hp: dict, pretrained_state: dict, n_up: int, setup: str, train: TabularData,
             cards, cfg: ExpConfig, device, seed: int):
    model = build_model(arch, train.x_num.shape[1], cards, n_up, hp)
    model.load_state_dict(pretrained_state)
    replace_head(model, 1, mlp_head=setup.startswith("MLP"), hidden=cfg.mlp_head_dim)
    e2e = setup.endswith("E2E")
    if not e2e:
        freeze_feature_extractor(model)
    epochs, use_val = downstream_schedule(len(train), setup)
    val = None
    if use_val:
        train, val = _split_val(train, cfg.downstream_val_frac, seed)
        epochs = cfg.max_downstream_epochs
    model, _, _ = fit(model, train, val, epochs=epochs, lr=cfg.finetune_lr, weight_decay=hp["weight_decay"],
                      head_lr=cfg.head_lr, head_warmup_epochs=cfg.head_warmup_epochs if e2e else 0,
                      patience=cfg.downstream_patience, batch_size=cfg.batch_size, device=device, seed=seed)
    return model


# ============================================================================ from scratch
def train_scratch(arch: str, hp: dict, train: TabularData, cards, cfg: ExpConfig, device, seed: int,
                  n_epochs: int = None):
    """n_epochs: số epoch tốt nhất tìm được khi tune trên upstream-subsample (Phụ lục B.6.3).
    Nếu None (không tune) thì dùng lịch giống E2E."""
    model = build_model(arch, train.x_num.shape[1], cards, 1, hp)
    val = None
    if n_epochs is None:
        n_epochs, use_val = downstream_schedule(len(train), "FS")
        if use_val:
            train, val = _split_val(train, cfg.downstream_val_frac, seed)
            n_epochs = cfg.max_downstream_epochs
    lr = hp.get("lr", cfg.scratch_lr)
    model, _, _ = fit(model, train, val, epochs=n_epochs, lr=lr, weight_decay=hp["weight_decay"],
                      patience=cfg.downstream_patience, batch_size=cfg.batch_size, device=device, seed=seed)
    return model

In [ ]:
%%writefile gbdt.py
"""
gbdt.py
CatBoost / XGBoost baseline, có và không có Stacking (Mục 3.3, Phụ lục B.6.4):
  - XGBoost: 11 bộ phân loại nhị phân trên 11 nhãn upstream
  - CatBoost: 1 mô hình MultiLogloss dự đoán cả 11 nhãn upstream
  Xác suất dự đoán được nối thêm vào đặc trưng downstream.
"""
import numpy as np


def make_gbdt(kind: str, params: dict, seed: int):
    if kind == "xgboost":
        from xgboost import XGBClassifier
        return XGBClassifier(**params, random_state=seed, tree_method="hist", n_jobs=-1, eval_metric="logloss")
    if kind == "catboost":
        from catboost import CatBoostClassifier
        p = dict(params)
        if "bagging_temperature" in p:
            p.setdefault("bootstrap_type", "Bayesian")
        return CatBoostClassifier(**p, random_seed=seed, verbose=0, thread_count=-1, allow_writing_files=False)
    raise ValueError(kind)


def fit_predict_gbdt(kind, params, X_tr, y_tr, X_te, seed):
    model = make_gbdt(kind, params, seed)
    model.fit(X_tr, y_tr.astype(int))
    return model.predict_proba(X_te)[:, 1]


class StackingFeatures:
    """Huấn luyện các mô hình GBDT upstream một lần cho mỗi downstream target, sau đó sinh
    xác suất của các nhãn upstream để nối vào dữ liệu downstream."""

    def __init__(self, kind: str, X_up: np.ndarray, Y_up: np.ndarray, seed: int = 0):
        self.kind = kind
        if kind == "xgboost":
            from xgboost import XGBClassifier
            self.models = [XGBClassifier(random_state=seed, tree_method="hist", n_jobs=-1,
                                         eval_metric="logloss").fit(X_up, Y_up[:, j].astype(int))
                           for j in range(Y_up.shape[1])]
        elif kind == "catboost":
            from catboost import CatBoostClassifier
            self.model = CatBoostClassifier(loss_function="MultiLogloss", random_seed=seed, verbose=0,
                                            thread_count=-1, allow_writing_files=False)
            self.model.fit(X_up, Y_up.astype(int))
        else:
            raise ValueError(kind)

    def transform(self, X: np.ndarray) -> np.ndarray:
        if self.kind == "xgboost":
            probs = np.column_stack([m.predict_proba(X)[:, 1] for m in self.models])
        else:
            probs = np.asarray(self.model.predict_proba(X))
        return np.hstack([X, probs]).astype(np.float32)

In [ ]:
%%writefile tuning.py
"""
tuning.py
Tối ưu siêu tham số bằng Optuna (TPE / Bayesian), 50 trials (Mục 3.4, Phụ lục C).
  - Mô hình transfer: tune trên TOÀN BỘ upstream, chọn theo upstream-val (multi-label AUC)
  - FS & GBDT: tune trên 1 nhãn upstream ngẫu nhiên với SỐ MẪU = số mẫu downstream,
    đánh giá trên upstream-val đầy đủ; FS lưu thêm epoch tốt nhất.
"""
import numpy as np
import optuna

from config import DEFAULT_HPARAMS, ExpConfig
from gbdt import fit_predict_gbdt
from models import build_model
from training import auc, fit

optuna.logging.set_verbosity(optuna.logging.WARNING)


def _maybe_zero(trial, name, low, high, log=True):
    return trial.suggest_float(name, low, high, log=log) if trial.suggest_categorical(f"use_{name}", [True, False]) else 0.0


def suggest_deep(trial, arch: str) -> dict:
    if arch == "ft_transformer":                               # Bảng 2
        return dict(n_layers=trial.suggest_int("n_layers", 1, 4),
                    d_token=trial.suggest_int("d_token", 64, 512, step=8),   # chia hết cho 8 head
                    residual_dropout=_maybe_zero(trial, "residual_dropout", 0.0, 0.2, log=False),
                    attention_dropout=trial.suggest_float("attention_dropout", 0.0, 0.5),
                    ffn_dropout=trial.suggest_float("ffn_dropout", 0.0, 0.5),
                    d_ffn_factor=trial.suggest_float("d_ffn_factor", 2 / 3, 8 / 3),
                    n_heads=8,
                    lr=trial.suggest_float("lr", 1e-5, 1e-3, log=True),
                    weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True))
    if arch == "resnet":                                       # Bảng 3
        return dict(n_layers=trial.suggest_int("n_layers", 1, 8),
                    d_embedding=trial.suggest_int("d_embedding", 64, 512),
                    d=trial.suggest_int("d", 64, 512),
                    d_hidden_factor=trial.suggest_float("d_hidden_factor", 1.0, 4.0),
                    hidden_dropout=trial.suggest_float("hidden_dropout", 0.0, 0.5),
                    residual_dropout=_maybe_zero(trial, "residual_dropout", 0.0, 0.5, log=False),
                    lr=trial.suggest_float("lr", 1e-5, 1e-2, log=True),
                    weight_decay=_maybe_zero(trial, "weight_decay", 1e-6, 1e-3))
    if arch == "mlp":                                          # Bảng 4 (kiểu RTDL: first/middle/last)
        n = trial.suggest_int("n_layers", 1, 8)
        first = trial.suggest_int("d_first", 1, 512)
        middle = trial.suggest_int("d_middle", 1, 512) if n > 2 else first
        last = trial.suggest_int("d_last", 1, 512) if n > 1 else first
        layers = [first] + [middle] * max(n - 2, 0) + ([last] if n > 1 else [])
        return dict(d_layers=layers, d_embedding=trial.suggest_int("d_embedding", 64, 512),
                    dropout=_maybe_zero(trial, "dropout", 0.0, 0.5, log=False),
                    lr=trial.suggest_float("lr", 1e-5, 1e-2, log=True),
                    weight_decay=_maybe_zero(trial, "weight_decay", 1e-6, 1e-3))
    if arch == "tab_transformer":                              # Bảng 5
        return dict(n_heads=trial.suggest_int("n_heads", 2, 8),
                    n_layers=trial.suggest_int("n_layers", 1, 12),
                    d_embedding=trial.suggest_int("d_embedding", 8, 128),
                    attention_dropout=trial.suggest_float("attention_dropout", 0.0, 0.5),
                    ff_dropout=trial.suggest_float("ff_dropout", 0.0, 0.5),
                    lr=trial.suggest_float("lr", 1e-6, 1e-3, log=True),
                    weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True))
    raise ValueError(arch)


def suggest_gbdt(trial, kind: str) -> dict:
    if kind == "catboost":                                     # Bảng 6
        return dict(iterations=trial.suggest_int("iterations", 2, 1000),
                    depth=trial.suggest_int("depth", 3, 10),
                    learning_rate=trial.suggest_float("learning_rate", 1e-5, 1.0, log=True),
                    bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 1.0),
                    l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
                    leaf_estimation_iterations=trial.suggest_int("leaf_estimation_iterations", 1, 10))
    if kind == "xgboost":                                      # Bảng 7
        return dict(n_estimators=trial.suggest_int("n_estimators", 2, 1000),
                    max_depth=trial.suggest_int("max_depth", 3, 10),
                    min_child_weight=trial.suggest_float("min_child_weight", 1e-8, 1e5, log=True),
                    subsample=trial.suggest_float("subsample", 0.5, 1.0),
                    learning_rate=trial.suggest_float("learning_rate", 1e-5, 1.0, log=True),
                    colsample_bylevel=trial.suggest_float("colsample_bylevel", 0.5, 1.0),
                    colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
                    gamma=_maybe_zero(trial, "gamma", 1e-8, 1e2),
                    reg_lambda=_maybe_zero(trial, "reg_lambda", 1e-8, 1e2),
                    reg_alpha=_maybe_zero(trial, "reg_alpha", 1e-8, 1e2))
    raise ValueError(kind)


def _study(seed):
    return optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))


def _safe(x):
    return -1.0 if (x is None or np.isnan(x)) else x


# ---------------------------------------------------------------- transfer: tune trên toàn bộ upstream
def tune_pretrain(arch, up, cards, cfg: ExpConfig, device, seed=0):
    def objective(trial):
        hp = suggest_deep(trial, arch)
        model = build_model(arch, up["train"].x_num.shape[1], cards, up["train"].y.shape[1], hp)
        _, _, score = fit(model, up["train"], up["val"], epochs=cfg.pretrain_epochs, lr=hp["lr"],
                          weight_decay=hp["weight_decay"], patience=cfg.pretrain_patience,
                          batch_size=cfg.batch_size, device=device, seed=seed)
        trial.set_user_attr("hp", hp)
        return _safe(score)

    st = _study(seed)
    st.optimize(objective, n_trials=cfg.n_trials)
    return st.best_trial.user_attrs["hp"], st.best_value


# ---------------------------------------------------------------- FS: tune trên upstream-subsample
def tune_scratch(arch, proxy, cards, cfg: ExpConfig, device, seed=0):
    def objective(trial):
        hp = suggest_deep(trial, arch)
        model = build_model(arch, proxy["train"].x_num.shape[1], cards, 1, hp)
        _, best_ep, score = fit(model, proxy["train"], proxy["val"], epochs=cfg.max_downstream_epochs,
                                lr=hp["lr"], weight_decay=hp["weight_decay"], patience=cfg.downstream_patience,
                                batch_size=cfg.batch_size, device=device, seed=seed)
        trial.set_user_attr("hp", hp)
        trial.set_user_attr("best_epoch", int(best_ep))
        return _safe(score)

    st = _study(seed)
    st.optimize(objective, n_trials=cfg.n_trials)
    return st.best_trial.user_attrs["hp"], st.best_trial.user_attrs["best_epoch"]


# ---------------------------------------------------------------- GBDT: tune trên upstream-subsample
def tune_gbdt(kind, proxy, cfg: ExpConfig, seed=0):
    def objective(trial):
        params = suggest_gbdt(trial, kind)
        try:
            p = fit_predict_gbdt(kind, params, proxy["train"].x_gbdt, proxy["train"].y, proxy["val"].x_gbdt, seed)
            return _safe(auc(proxy["val"].y, p))
        except Exception:
            return -1.0

    st = _study(seed)
    st.optimize(objective, n_trials=cfg.n_trials)
    params = suggest_gbdt(optuna.trial.FixedTrial(st.best_params), kind)
    return params


def default_hp(arch):
    return dict(DEFAULT_HPARAMS[arch])

In [ ]:
%%writefile run_experiments.py
"""
run_experiments.py
Chạy toàn bộ thực nghiệm Mục 4 (Hình 2) của bài báo:
  12 nhiệm vụ downstream x 5 mức dữ liệu (4/10/20/100/200) = 60 kịch bản
  Mô hình sâu: FS + 4 kịch bản transfer (LH, MLP, LH-E2E, MLP-E2E)
  GBDT: CatBoost/XGBoost có & không Stacking
  Mỗi cấu hình chạy với 10 seeds, đo ROC-AUC trên tập test downstream cố định.

Ví dụ:
  python run_experiments.py --dataset metamimic --data-path data/MetaMIMIC.csv
  python run_experiments.py --dataset metamimic --data-path data/MetaMIMIC.csv --quick   # chạy thử
  python run_experiments.py --dataset yeast --models ft_transformer mlp catboost   # tự tải Yeast từ OpenML
"""
import argparse
import json
import os
import time

import numpy as np
import pandas as pd
import torch

from config import DEEP_MODELS, GBDT_MODELS, TRANSFER_SETUPS, ExpConfig, quick_config
from data import TransferBenchmark, load_metamimic, load_mulan, load_openml_yeast
from gbdt import StackingFeatures, fit_predict_gbdt
from training import auc, finetune, predict_logits, pretrain, train_scratch
from tuning import default_hp, tune_gbdt, tune_pretrain, tune_scratch

RESULT_COLS = ["dataset", "target_idx", "target", "n_samples", "model", "setup", "seed", "auc"]


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--dataset", default="metamimic", choices=["metamimic", "yeast", "emotions", "mulan"])
    p.add_argument("--data-path", default="openml", help="đường dẫn CSV/ARFF; với yeast có thể để 'openml' để tự tải")
    p.add_argument("--n-labels", type=int, default=None, help="số cột nhãn cuối file (Yeast=14, Emotions=6)")
    p.add_argument("--models", nargs="+", default=DEEP_MODELS + GBDT_MODELS)
    p.add_argument("--setups", nargs="+", default=["FS"] + TRANSFER_SETUPS + ["Stacking"])
    p.add_argument("--targets", nargs="+", type=int, default=None, help="chỉ số nhiệm vụ downstream (mặc định: tất cả)")
    p.add_argument("--sizes", nargs="+", type=int, default=None, help="vd: 4 10 20 100 200 (hoặc 400 1000 2000)")
    p.add_argument("--seeds", type=int, default=None, help="số seed (mặc định 10)")
    p.add_argument("--n-trials", type=int, default=None)
    p.add_argument("--no-tune", action="store_true", help="dùng siêu tham số mặc định (Bảng 2-5)")
    p.add_argument("--pretrain-per-seed", action="store_true")
    p.add_argument("--quick", action="store_true", help="cấu hình rút gọn để chạy thử")
    p.add_argument("--batch-size", type=int, default=None, help="mặc định 256 như bài báo; giảm nếu thiếu bộ nhớ")
    p.add_argument("--out-dir", default="outputs")
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    return p.parse_args()


def build_cfg(a) -> ExpConfig:
    cfg = quick_config() if a.quick else ExpConfig()
    if a.sizes: cfg.sample_sizes = a.sizes
    if a.seeds: cfg.seeds = list(range(a.seeds))
    if a.n_trials: cfg.n_trials = a.n_trials
    if a.no_tune: cfg.tune = False
    if a.pretrain_per_seed: cfg.pretrain_per_seed = True
    if a.batch_size: cfg.batch_size = a.batch_size
    return cfg


def load_benchmark(a, cfg):
    if a.dataset == "metamimic":
        X, Y, num, cat = load_metamimic(a.data_path)
    elif a.dataset == "yeast" and a.data_path == "openml":
        X, Y, num, cat = load_openml_yeast()
    else:
        n_labels = a.n_labels or {"yeast": 14, "emotions": 6}.get(a.dataset)
        if n_labels is None:
            raise ValueError("Cần --n-labels")
        X, Y, num, cat = load_mulan(a.data_path, n_labels)
    return TransferBenchmark(X, Y, num, cat, cfg, name=a.dataset)


class Cache:
    """Lưu siêu tham số / trọng số pre-train ra đĩa để có thể chạy lại, tiếp tục giữa chừng."""

    def __init__(self, root):
        self.root = root
        os.makedirs(root, exist_ok=True)

    def json(self, key, fn):
        path = os.path.join(self.root, key + ".json")
        if os.path.exists(path):
            return json.load(open(path))
        val = fn()
        json.dump(val, open(path, "w"), indent=2)
        return val

    def torch(self, key, fn):
        path = os.path.join(self.root, key + ".pt")
        if os.path.exists(path):
            return torch.load(path, map_location="cpu")
        val = fn()
        torch.save(val, path)
        return val


def main():
    a = parse_args()
    cfg = build_cfg(a)
    torch.set_num_threads(max(1, os.cpu_count() or 1))
    bench = load_benchmark(a, cfg)
    cache = Cache(os.path.join(a.out_dir, "cache", a.dataset))
    res_path = os.path.join(a.out_dir, f"results_{a.dataset}.csv")
    done = pd.read_csv(res_path) if os.path.exists(res_path) else pd.DataFrame(columns=RESULT_COLS)
    done_keys = set(zip(done.target_idx, done.n_samples, done.model, done.setup, done.seed))
    rows = []

    def flush():
        nonlocal rows
        if rows:
            pd.DataFrame(rows, columns=RESULT_COLS).to_csv(res_path, mode="a", header=not os.path.exists(res_path),
                                                           index=False)
            rows = []

    def record(t, n, model, setup, seed, score):
        rows.append([a.dataset, t, bench.target_names[t], n, model, setup, seed, score])
        print(f"  t={t:2d} n={n:4d} {model:15s} {setup:8s} seed={seed} AUC={score:.4f}")

    deep = [m for m in a.models if m in DEEP_MODELS]
    gbdts = [m for m in a.models if m in GBDT_MODELS]
    transfer_setups = [s for s in a.setups if s in TRANSFER_SETUPS]
    targets = a.targets if a.targets is not None else list(range(bench.n_targets))
    cards, dev = bench.cat_cardinalities, a.device

    for t in targets:
        t0 = time.time()
        print(f"\n===== Downstream target {t}: {bench.target_names[t]} =====")
        up = bench.upstream(t)
        n_up = up["train"].y.shape[1]
        rng = np.random.RandomState(1000 + t)
        proxy_target = int(rng.choice(bench.upstream_targets(t)))   # nhãn upstream dùng để tune FS/GBDT

        # ---------- 1) Pre-training (tune trên toàn bộ upstream)
        pre_hp, pre_states = {}, {}
        if transfer_setups:
            for arch in deep:
                pre_hp[arch] = cache.json(f"hp_pretrain_{arch}_t{t}",
                                          lambda: tune_pretrain(arch, up, cards, cfg, dev)[0]
                                          if cfg.tune else default_hp(arch))
                pre_seeds = cfg.seeds if cfg.pretrain_per_seed else [0]
                for s in pre_seeds:
                    def _pt(arch=arch, s=s):
                        m, v, ep = pretrain(arch, pre_hp[arch], up, cards, cfg, dev, seed=s)
                        print(f"  [pretrain] {arch} seed={s}: upstream val AUC={v:.4f} (best epoch {ep})")
                        return {k: x.cpu() for k, x in m.state_dict().items()}
                    pre_states[(arch, s)] = cache.torch(f"pretrained_{arch}_t{t}_s{s}", _pt)

        stackers = {}
        for n in cfg.sample_sizes:
            down = bench.downstream(t, n)
            tr, te = down["train"], down["test"]

            # ---------- 2) Mô hình sâu
            for arch in deep:
                if "FS" in a.setups:
                    if cfg.tune:
                        hp_fs, ep_fs = cache.json(f"hp_fs_{arch}_t{t}_n{n}", lambda: list(tune_scratch(
                            arch, bench.upstream_single(proxy_target, n, seed=0), cards, cfg, dev)))
                    else:
                        hp_fs, ep_fs = default_hp(arch), None
                    for s in cfg.seeds:
                        if (t, n, arch, "FS", s) in done_keys: continue
                        m = train_scratch(arch, hp_fs, tr, cards, cfg, dev, seed=s, n_epochs=ep_fs)
                        record(t, n, arch, "FS", s, auc(te.y, predict_logits(m, te, dev)))
                for setup in transfer_setups:
                    for s in cfg.seeds:
                        if (t, n, arch, setup, s) in done_keys: continue
                        state = pre_states[(arch, s if cfg.pretrain_per_seed else 0)]
                        m = finetune(arch, pre_hp[arch], state, n_up, setup, tr, cards, cfg, dev, seed=s)
                        record(t, n, arch, setup, s, auc(te.y, predict_logits(m, te, dev)))
                flush()

            # ---------- 3) GBDT (FS và Stacking)
            for kind in gbdts:
                params = cache.json(f"hp_{kind}_t{t}_n{n}", lambda: tune_gbdt(
                    kind, bench.upstream_single(proxy_target, n, seed=0), cfg) if cfg.tune else default_hp(kind))
                if "Stacking" in a.setups:
                    if kind not in stackers:
                        stackers[kind] = StackingFeatures(kind, up["train"].x_gbdt, up["train"].y)
                    st = stackers[kind]
                    Xtr_st, Xte_st = st.transform(tr.x_gbdt), st.transform(te.x_gbdt)
                for s in cfg.seeds:
                    if "FS" in a.setups and (t, n, kind, "FS", s) not in done_keys:
                        p = fit_predict_gbdt(kind, params, tr.x_gbdt, tr.y, te.x_gbdt, s)
                        record(t, n, kind, "FS", s, auc(te.y, p))
                    if "Stacking" in a.setups and (t, n, kind, "Stacking", s) not in done_keys:
                        p = fit_predict_gbdt(kind, params, Xtr_st, tr.y, Xte_st, s)
                        record(t, n, kind, "Stacking", s, auc(te.y, p))
                flush()
        print(f"  -> xong target {t} trong {(time.time() - t0) / 60:.1f} phút")
    flush()
    print(f"\nKết quả lưu tại {res_path}. Chạy: python analyze_results.py --results {res_path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analyze_results.py
"""
analyze_results.py
Tổng hợp kết quả giống Hình 2, 7, 9 của bài báo:
  - Thứ hạng có xét ý nghĩa thống kê: trên mỗi (nhiệm vụ, số mẫu), hạng của mô hình m
    = 1 + số mô hình tốt hơn m một cách có ý nghĩa (Wilcoxon rank-sum một phía, p < 0.05,
    trên ROC-AUC của các seed). Các mô hình top không khác biệt có ý nghĩa đều nhận hạng 1.
  - Trung bình hạng trên 12 nhiệm vụ -> heatmap; sai số chuẩn của hạng; AUC trung bình;
    mức cải thiện ROC-AUC so với CatBoost.

  python analyze_results.py --results outputs/results_metamimic.csv
"""
import argparse
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ranksums

MODEL_ORDER = ["ft_transformer", "resnet", "mlp", "tab_transformer", "catboost", "xgboost"]
MODEL_LABEL = {"ft_transformer": "FT-Transformer", "resnet": "ResNet", "mlp": "MLP",
               "tab_transformer": "TabTransformer", "catboost": "CatBoost", "xgboost": "XGBoost"}
SETUP_ORDER = ["FS", "LH-E2E", "MLP-E2E", "LH", "MLP", "Stacking"]


def significance_ranks(df: pd.DataFrame, alpha=0.05) -> pd.DataFrame:
    out = []
    for (t, n), g in df.groupby(["target_idx", "n_samples"]):
        scores = {k: v["auc"].dropna().values for k, v in g.groupby(["model", "setup"])}
        for m, sm in scores.items():
            better = 0
            for o, so in scores.items():
                if o == m or len(so) < 2 or len(sm) < 2:
                    continue
                if ranksums(so, sm, alternative="greater").pvalue < alpha:
                    better += 1
            out.append(dict(target_idx=t, n_samples=n, model=m[0], setup=m[1], rank=1 + better,
                            mean_auc=float(np.mean(sm))))
    return pd.DataFrame(out)


def ordered_columns(cols):
    return sorted(cols, key=lambda c: (MODEL_ORDER.index(c[0]) if c[0] in MODEL_ORDER else 99,
                                       SETUP_ORDER.index(c[1]) if c[1] in SETUP_ORDER else 99))


def plot_panels(table: pd.DataFrame, path: str, title: str, cmap="RdYlBu", fmt=".1f", cbar_label="Rank"):
    table = table.dropna(axis=1, how="all")
    models = [m for m in MODEL_ORDER if m in table.columns.get_level_values(0)]
    if not models:
        print(f"  (bỏ qua {os.path.basename(path)}: chưa đủ dữ liệu, ví dụ cần >= 2 nhiệm vụ để tính SE)")
        return
    widths = [table[m].shape[1] for m in models]
    fig, axes = plt.subplots(1, len(models), figsize=(1.0 * sum(widths) + 2, 3.6),
                             gridspec_kw={"width_ratios": widths}, squeeze=False)
    vmin, vmax = np.nanmin(table.values), np.nanmax(table.values)
    im = None
    for ax, m in zip(axes[0], models):
        sub = table[m]
        im = ax.imshow(sub.values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
        for i in range(sub.shape[0]):
            for j in range(sub.shape[1]):
                ax.text(j, i, format(sub.values[i, j], fmt), ha="center", va="center", fontsize=8)
        ax.set_xticks(range(sub.shape[1]))
        ax.set_xticklabels(sub.columns, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(sub.shape[0]))
        ax.set_yticklabels(sub.index if ax is axes[0][0] else [], fontsize=8)
        ax.set_title(MODEL_LABEL.get(m, m), fontsize=9)
    axes[0][0].set_ylabel("Num samples")
    fig.colorbar(im, ax=axes[0].tolist(), label=cbar_label, shrink=0.8)
    fig.suptitle(title, fontsize=10)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", required=True)
    ap.add_argument("--out-dir", default=None)
    ap.add_argument("--alpha", type=float, default=0.05)
    ap.add_argument("--exclude", nargs="*", default=[], help="vd: tab_transformer (Hình 2 không có TabTransformer)")
    a = ap.parse_args()
    out = a.out_dir or os.path.join(os.path.dirname(a.results) or ".", "analysis")
    os.makedirs(out, exist_ok=True)

    df = pd.read_csv(a.results)
    df = df[~df.model.isin(a.exclude)]
    ranks = significance_ranks(df, a.alpha)
    ranks.to_csv(os.path.join(out, "ranks_per_task.csv"), index=False)

    def pivot(values, agg):
        t = ranks.pivot_table(index="n_samples", columns=["model", "setup"], values=values, aggfunc=agg)
        t = t[ordered_columns(list(t.columns))]
        return t.sort_index(ascending=False)

    avg_rank = pivot("rank", "mean")
    se = ranks.groupby(["n_samples", "model", "setup"])["rank"].agg(
        lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan)
    se_rank = se.unstack(["model", "setup"]).reindex(columns=avg_rank.columns).sort_index(ascending=False)
    mean_auc = pivot("mean_auc", "mean")
    avg_rank.round(2).to_csv(os.path.join(out, "avg_rank.csv"))
    se_rank.round(2).to_csv(os.path.join(out, "rank_std_error.csv"))
    mean_auc.round(4).to_csv(os.path.join(out, "mean_auc.csv"))

    plot_panels(avg_rank, os.path.join(out, "fig2_avg_ranks.png"),
                "Average rank across downstream tasks (lower = better)")
    plot_panels(se_rank, os.path.join(out, "fig7_rank_std_error.png"), "Rank standard error",
                cmap="Blues_r", cbar_label="Rank SE")

    # Hình 9: cải thiện ROC-AUC so với CatBoost (FS), trung bình trên các nhiệm vụ
    if ("catboost", "FS") in set(zip(ranks.model, ranks.setup)):
        base = ranks[(ranks.model == "catboost") & (ranks.setup == "FS")][["target_idx", "n_samples", "mean_auc"]]
        imp = ranks.merge(base, on=["target_idx", "n_samples"], suffixes=("", "_cb"))
        imp["delta"] = imp["mean_auc"] - imp["mean_auc_cb"]
        imp_t = imp.pivot_table(index="n_samples", columns=["model", "setup"], values="delta", aggfunc="mean")
        imp_t = imp_t[ordered_columns(list(imp_t.columns))].sort_index(ascending=False)
        imp_t.round(3).to_csv(os.path.join(out, "auc_improvement_over_catboost.csv"))
        plot_panels(imp_t, os.path.join(out, "fig9_auc_improvement.png"),
                    "Average ROC-AUC improvement over CatBoost", cmap="RdYlBu_r", fmt=".2f",
                    cbar_label="ΔAUC")

    pd.set_option("display.width", 250)
    print("\nAverage rank (hàng = số mẫu downstream):\n", avg_rank.round(1))
    print(f"\nĐã lưu bảng & hình vào {out}/")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile make_synthetic_data.py
"""
make_synthetic_data.py
Sinh file CSV GIẢ LẬP có cùng cấu trúc MetaMIMIC (subject_id, 171 cột số, gender, 12 cột *_diagnosed,
có giá trị thiếu) để kiểm tra pipeline chạy được khi chưa có quyền truy cập MIMIC-IV.
KHÔNG dùng để báo cáo kết quả.

  python make_synthetic_data.py --n 3000 --out data/synthetic_metamimic.csv
"""
import argparse
import os

import numpy as np
import pandas as pd

from config import MIMIC_TARGETS


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=3000)
    ap.add_argument("--out", default="data/synthetic_metamimic.csv")
    a = ap.parse_args()
    rng = np.random.default_rng(0)
    n, d = a.n, 171
    Z = rng.normal(size=(n, 8))                                # yếu tố ẩn chung -> các nhãn có tương quan
    X = Z @ rng.normal(size=(8, d)) + rng.normal(scale=1.0, size=(n, d))
    X = np.exp(X / 4) * 50                                     # phân phối lệch giống xét nghiệm
    X[rng.random(X.shape) < 0.05] = np.nan
    df = pd.DataFrame(X, columns=[f"lab_{i}" for i in range(d)])
    df.insert(0, "subject_id", np.arange(n))
    df["gender"] = rng.integers(0, 2, n).astype(float)
    W = rng.normal(size=(8, 12))
    prev = np.array([.25, .6, .33, .19, .1, .36, .2, .15, .4, .1, .12, .08])
    logits = Z @ W + rng.normal(size=(n, 12))
    for j, c in enumerate(MIMIC_TARGETS):
        df[c] = (logits[:, j] > np.quantile(logits[:, j], 1 - prev[j])).astype(int)
    os.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)
    df.to_csv(a.out, index=False)
    print(f"Saved {a.out}: {df.shape}")


if __name__ == "__main__":
    main()

## 3. Chạy thực nghiệm — chọn MỘT trong ba lựa chọn

Tham số hay dùng của `run_experiments.py`:

| Tham số | Ý nghĩa |
|---|---|
| `--quick` | Cấu hình rút gọn (3 seed, 2 trial Optuna, ít epoch) để chạy thử |
| `--no-tune` | Bỏ Optuna, dùng siêu tham số mặc định của bài báo |
| `--targets 0 1` | Chỉ chạy một số nhiệm vụ downstream (mặc định: tất cả) |
| `--sizes 4 20 100` | Số mẫu downstream (mặc định: 4 10 20 100 200) |
| `--models ...` | `ft_transformer resnet mlp tab_transformer catboost xgboost` |
| `--seeds 10` | Số seed (bài báo: 10) |
| `--batch-size 128` | Giảm nếu bị lỗi hết bộ nhớ GPU |

### Lựa chọn A — Dữ liệu giả lập (chạy thử pipeline, ~10–20 phút)
Dữ liệu giả có cấu trúc giống MetaMIMIC (171 cột số + gender, 12 nhãn bệnh). **Kết quả không có ý nghĩa khoa học**, chỉ để kiểm tra pipeline chạy đúng.

In [ ]:
DATASET = "metamimic"
!python make_synthetic_data.py --n 3000 --out data/synthetic_metamimic.csv
!python run_experiments.py --dataset metamimic --data-path data/synthetic_metamimic.csv \
    --quick --targets 0 --models ft_transformer mlp catboost xgboost --out-dir outputs_synthetic
RESULTS = "outputs_synthetic/results_metamimic.csv"

### Lựa chọn B — Yeast thật (Phụ lục D.6, tự tải từ OpenML)
2.417 gen, 103 đặc trưng, 14 nhãn → 14 phân chia upstream/downstream. Bài báo dùng FT-Transformer, MLP và CatBoost cho thí nghiệm này.
Bản dưới đây dùng `--no-tune` và 3 nhiệm vụ để chạy nhanh; bỏ `--targets` và `--no-tune` để chạy đầy đủ.

In [ ]:
DATASET = "yeast"
!python run_experiments.py --dataset yeast --data-path openml \
    --models ft_transformer mlp catboost --setups FS LH MLP LH-E2E MLP-E2E Stacking \
    --no-tune --targets 0 1 2 --seeds 5 --out-dir outputs_yeast
RESULTS = "outputs_yeast/results_yeast.csv"

### Lựa chọn C — MetaMIMIC thật (thực nghiệm chính, Mục 4)
Cần quyền truy cập MIMIC-IV (PhysioNet) và tự tạo `MetaMIMIC.csv` theo `github.com/ModelOriented/metaMIMIC`.
Tải file lên Google Drive rồi sửa đường dẫn bên dưới.

Bản đầy đủ (Optuna 50 trial, 10 seed, 12 nhiệm vụ) rất nặng, vượt quá giới hạn một phiên Colab. Nên chạy theo nấc:
`--no-tune` cho 1 nhiệm vụ → `--no-tune` cho cả 12 nhiệm vụ → bản có tune. Nếu phiên bị ngắt, chạy lại đúng ô này để tiếp tục.

In [ ]:
DATASET = "metamimic"
MIMIC_PATH = "/content/drive/MyDrive/MetaMIMIC.csv"   # <-- sửa đường dẫn

# Kiểm tra nhanh cấu trúc file
import pandas as pd
df = pd.read_csv(MIMIC_PATH)
print(df.shape, "| số cột *_diagnosed:", sum(c.endswith("_diagnosed") for c in df.columns))
print(df.filter(like="_diagnosed").mean().round(3))   # so với tần suất ở Bảng 1 của bài báo
del df

!python run_experiments.py --dataset metamimic --data-path "$MIMIC_PATH" \
    --no-tune --targets 0 --out-dir outputs_mimic
RESULTS = "outputs_mimic/results_metamimic.csv"

## 4. Phân tích kết quả (giống Hình 2, 7, 9)

In [ ]:
!python analyze_results.py --results $RESULTS --exclude tab_transformer

import os, glob
from IPython.display import Image, display
ana = os.path.join(os.path.dirname(RESULTS), "analysis")
for f in sorted(glob.glob(os.path.join(ana, "fig*.png"))):
    print(os.path.basename(f)); display(Image(f))

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50)
print("Thứ hạng trung bình (thấp hơn = tốt hơn):")
display(pd.read_csv(os.path.join(ana, "avg_rank.csv"), header=[0, 1], index_col=0).round(1))
print("ROC-AUC trung bình:")
display(pd.read_csv(os.path.join(ana, "mean_auc.csv"), header=[0, 1], index_col=0).round(3))

### Cách đọc kết quả
- Hàng = số mẫu downstream; cột = mô hình × kịch bản. `FS` = huấn luyện từ đầu, `LH`/`MLP` = head tuyến tính/MLP trên feature extractor đóng băng, `-E2E` = fine-tune toàn bộ, `Stacking` = GBDT có nối dự đoán upstream.
- Hạng của một cấu hình = 1 + số cấu hình tốt hơn nó **có ý nghĩa thống kê** (Wilcoxon rank-sum một phía trên ROC-AUC của các seed), trung bình trên các nhiệm vụ.
- Xu hướng bài báo báo cáo trên MetaMIMIC: FT-Transformer E2E tốt hơn mọi GBDT ở mọi mức dữ liệu; MLP với head MLP mạnh nhất ở 4–10 mẫu; Stacking giúp GBDT nhưng vẫn thua transfer learning.

## 5. (Tùy chọn) Tải kết quả về máy

In [ ]:
!zip -qr results_export.zip $(dirname $RESULTS)
from google.colab import files
files.download("results_export.zip")